# PrimeVul Score Conversion (Multiple Choice)

Converts lm-eval loglikelihood output (primevul_choice_1to2) into router training data.
Same structure as convert_dataset_7_model.ipynb (MMLU/ARC section).

Paper Eq. 2: score = softmax(log_probs)[target]  if acc == 1  else  0

In [33]:
import json
import os
import glob
import random
import numpy as np
import pandas as pd

random.seed(42)

output_file_path = "./datasets/split2_primevul_choice"
os.makedirs(output_file_path, exist_ok=True)

# [org, model_name, output_dir_name]
model_list = [
    ["codellama",      "CodeLlama-7b-Instruct-hf",        "CodeLlama-7b-Instruct"],
    ["codellama",      "CodeLlama-13b-Instruct-hf",       "CodeLlama-13b-Instruct"],
    ["deepseek-ai",    "DeepSeek-Coder-V2-Lite-Instruct", "DeepSeek-Coder-V2-Lite-Instruct"],
    ["Qwen",           "Qwen2.5-Coder-14B-Instruct",      "Qwen2.5-Coder-14B-Instruct"],
    ["bigcode",        "starcoder2-15b-instruct-v0.1",    "starcoder2-15b-instruct"],
    ["Virtue-AI-HUB",  "VulnLLM-R-7B",                   "VulnLLM-R-7B"],
]

# PrimeVul (loglikelihood / multiple choice)

In [34]:
# scores_by_idx[idx][model_id] = score
# funcs_by_idx[idx]            = func text
scores_by_idx = {}
funcs_by_idx  = {}

for model_index, model_info in enumerate(model_list):
    model_pre  = model_info[0]
    model      = model_info[1]
    dir_name   = model_info[2]
    model_id   = model_pre + "/" + model

    pattern = f"./output/primevul_choice_1to2/{dir_name}/*/samples_primevul_choice_1to2_*.jsonl"
    files = glob.glob(pattern)
    if not files:
        print(f"[skip] {model}: no samples file at {pattern}")
        continue

    n_docs = 0
    avg_score = 0.0

    with open(files[0]) as f:
        for line in f:
            if not line.strip():
                continue
            entry  = json.loads(line)

            # Use doc["idx"] as stable key across model runs
            idx    = entry["doc"]["idx"]
            resps  = entry["resps"]  
            target = int(entry["target"])  # 0 or 1
            acc    = entry["acc"]          # 1.0 if correct, 0.0 if wrong

            # Paper Eq. 2: softmax over log-probs of each choice
            log_probs   = np.array([float(resps[c][0][0]) for c in range(len(resps))])
            # print(log_probs)
            probability = np.exp(log_probs)
            probability = probability / np.sum(probability)
            score = float(probability[target]) if acc == 1.0 else 0.0

            if idx not in scores_by_idx:
                scores_by_idx[idx] = {}
            scores_by_idx[idx][model_id] = score

            if idx not in funcs_by_idx:
                funcs_by_idx[idx] = entry["doc"]["func"]

            n_docs    += 1
            avg_score += 1 if acc == 1.0 else 0

    avg_score /= n_docs if n_docs else 1
    print(f"[ok] {model:<45} docs={n_docs}  avg_score={avg_score:.4f}")

# Build output_data sorted by idx for reproducibility
output_data = [
    {"question": funcs_by_idx[idx], "scores": scores_by_idx[idx]}
    for idx in sorted(scores_by_idx.keys())
]

print(f"\nTotal docs: {len(output_data)}")

[ok] CodeLlama-7b-Instruct-hf                      docs=18012  avg_score=0.3331
[ok] CodeLlama-13b-Instruct-hf                     docs=18012  avg_score=0.3354
[ok] DeepSeek-Coder-V2-Lite-Instruct               docs=18012  avg_score=0.3242
[ok] Qwen2.5-Coder-14B-Instruct                    docs=18012  avg_score=0.3900
[ok] starcoder2-15b-instruct-v0.1                  docs=18012  avg_score=0.3133
[ok] VulnLLM-R-7B                                  docs=18012  avg_score=0.5409

Total docs: 18012


In [35]:
# Train / test split  70 / 30  (same as paper)
train_split_index = random.sample(range(len(output_data)), len(output_data))
output_data = [output_data[idx] for idx in train_split_index]

train_split = output_data[:int(0.7 * len(output_data))]
test_split  = output_data[int(0.7 * len(output_data)):]

with open(os.path.join(output_file_path, "primevul_choice_train.json"), "w") as f:
    json.dump(train_split, f)

with open(os.path.join(output_file_path, "primevul_choice_test.json"), "w") as f:
    json.dump(test_split, f)

print(f"train: {len(train_split)}  test: {len(test_split)}")
print(f"Saved to {output_file_path}/")

train: 12608  test: 5404
Saved to ./datasets/split2_primevul_choice/


# Get ACC (per-model accuracy on train set)

In [36]:
with open(os.path.join(output_file_path, "primevul_choice_train.json")) as f:
    output_data = json.load(f)

correct_dict = {m[0]+"/"+m[1]: 0 for m in model_list}

data_size = len(output_data)
for item in output_data:
    for key, score in item["scores"].items():
        if key in correct_dict:
            correct_dict[key] += score / data_size * 100

df = pd.DataFrame.from_dict(correct_dict, orient="index", columns=["avg_score (%)"])
df

,avg_score (%)
codellama/CodeLlama-7b-Instruct-hf,27.199428
codellama/CodeLlama-13b-Instruct-hf,26.812735
deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct,31.606662
Qwen/Qwen2.5-Coder-14B-Instruct,34.759419
bigcode/starcoder2-15b-instruct-v0.1,27.260362
Virtue-AI-HUB/VulnLLM-R-7B,46.300362
